# Residual transformer experiments

This notebook documents the **causal residual transformer** module and recorded results on the Jubilant intermittent-demand slice (800 SKUs, ~90% zeros).

**Architecture contract**

Holiday and regressor are absorbed into `y_struct` (DeepSequence). The residual transformer only sees residual dynamics — it does **not** re-feed holiday/regressor features.

```
y_struct (DS: trend + seasonal + holiday + regressor)
        +
lookback sequence [y_struct, y, residual]
  (y & residual masked at predict step; windows built per SKU)
        │
        ▼
causal MultiHeadAttention × n_blocks
        │
        ▼
δ, p  →  ŷ = p · relu(y_struct + δ)
```

Full package docs: see `../README.md`.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from deepsequence_hierarchical_attention import (
    build_residual_transformer,
    build_residual_windows,
    train_residual_transformer,
    predict_residual_transformer,
    DEFAULT_SEQUENCE_CHANNELS,
)

print("channels:", DEFAULT_SEQUENCE_CHANNELS)
print("package root:", ROOT)

## 2. Synthetic mini-run (SKU-isolated windows)

Demonstrates that windows never cross SKU boundaries, then fits a tiny residual transformer.

In [ ]:
rng = np.random.default_rng(42)
rows = []
for sku in range(5):
    for t in range(40):
        y = float(rng.integers(0, 5) if rng.random() > 0.85 else 0)
        # Pretend holiday/regressor already live inside y_struct
        y_struct = 0.3 + 0.05 * (t % 7)
        rows.append(
            {
                "id_var": f"SKU_{sku}",
                "ds": pd.Timestamp("2024-01-01") + pd.Timedelta(days=t),
                "y": y,
                "y_struct": y_struct,
                "split": "train" if t < 30 else "val",
            }
        )
panel = pd.DataFrame(rows)

lookback = 14
X, y, y_struct, sku_ids, splits = build_residual_windows(panel, lookback=lookback)
sku_map = {k: i for i, k in enumerate(sorted(panel["id_var"].unique()))}
sku_idx = np.array([sku_map[s] for s in sku_ids], dtype=np.int32).reshape(-1, 1)

m_tr = splits == "train"
m_va = splits == "val"
print("X", X.shape, "channels=[y_struct,y,resid] train/val", m_tr.sum(), m_va.sum())
print("predict-step y/resid masked?", X[:, -1, 1].max(), X[:, -1, 2].max())

assert X.shape[1] == lookback and X.shape[-1] == 3
assert (X[:, -1, 1] == 0).all() and (X[:, -1, 2] == 0).all()

In [ ]:
model = build_residual_transformer(
    lookback=lookback,
    n_channels=X.shape[-1],
    n_skus=len(sku_map),
    d_model=16,
    n_heads=2,
    n_blocks=2,
)
model.summary()

zero_rate = float((y[m_tr] == 0).mean())
train_residual_transformer(
    model,
    X[m_tr], y[m_tr], y_struct[m_tr], sku_idx[m_tr],
    X[m_va], y[m_va], y_struct[m_va], sku_idx[m_va],
    zero_rate=zero_rate,
    epochs=3,
    batch_size=64,
    verbose=1,
)

yhat, p, base, delta = predict_residual_transformer(
    model, X[m_va], y_struct[m_va], sku_idx[m_va]
)
mae = float(np.mean(np.abs(y[m_va] - yhat)))
print(f"val MAE={mae:.3f}  mean_p={p.mean():.3f}  mean_delta={delta.mean():.3f}")

## 3. Recorded experiments (800 SKUs)

Loaded from JSON artifacts produced by `examples/eval_baselines_compare.py` and `examples/eval_residual_transformer.py`.

In [ ]:
def load_json(name):
    path = ROOT / name
    with open(path) as f:
        return json.load(f)

base = load_json("eval_results_baseline_compare_rounded.json")
rows = []
for name, blob in base["models"].items():
    t = blob["test"]
    r = t.get("rounded", {})
    rows.append(
        {
            "model": name,
            "mae_rounded": r.get("mae_all", t.get("mae_all")),
            "mae_nonzero": r.get("mae_nonzero", t.get("mae_nonzero")),
            "aucroc": t.get("aucroc"),
        }
    )
baseline_df = pd.DataFrame(rows).sort_values("mae_rounded")
print("Baseline compare (same 800-SKU causal slice)")
baseline_df

In [ ]:
def residual_table(path):
    data = load_json(path)
    cfg = data["config"]
    rows = []
    for name, blob in data["models"].items():
        t = blob["test"]
        rows.append(
            {
                "model": name,
                "mae_rounded": t.get("mae_all_rounded", t.get("mae_all")),
                "mae_nonzero": t.get("mae_nonzero"),
                "aucroc": t.get("aucroc"),
                "mean_final": t.get("mean_final"),
            }
        )
    df = pd.DataFrame(rows).sort_values("mae_rounded")
    meta = {
        "n_blocks": cfg.get("n_blocks", 1),
        "d_model": cfg.get("d_model", 32),
        "lookback": cfg.get("lookback"),
        "n_skus": cfg.get("n_skus"),
    }
    return meta, df

meta1, df1 = residual_table("eval_results_residual_transformer.json")
print("Residual TF — 1 block, d_model=32", meta1)
display(df1)

meta4, df4 = residual_table("eval_results_residual_transformer_nblocks4.json")
print("Residual TF — 4 blocks, d_model=64", meta4)
display(df4)

## 4. Conclusions from recorded runs

> Note: tables below were from an earlier residual setup that still fed calendar proxies into the TF head and used a no-lag structural base. The module/contract is now stricter: **holiday + regressor ∈ y_struct**, residual channels only.

1. **DS + lags + three_term** (~2.08 MAE rounded) beats residual stacks on this panel for all-day MAE, with strong nonzero MAE / AUROC.
2. Residual transformer recovers a lot vs a weak structural base, but should not re-learn holiday/regressor.
3. **Deeper blocks** helped freeze slightly in the old setup; E2E gains were small.
4. Short-term leftover signal after a full DS base is the right target for the residual head.

Reproduce (updated contract):

```bash
python examples/eval_baselines_compare.py
python examples/eval_residual_transformer.py --n_blocks 1 --d_model 32
python examples/eval_residual_transformer.py --n_blocks 4 --d_model 64 \
  --out_json eval_results_residual_transformer_nblocks4.json
```